# P4 — Task 4 — Segmentation + dataset nộp kèm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanh-tai-435/BI10GroupYAPPERS/blob/main/notebooks/task4_segmentation.ipynb)

**Slide sở hữu:** 14, 15 · **Slide mới:** +1: Kiểm định & dịch chuyển segment  
**Đã có:** task4.py (k-means k=4, silhouette 0.251), task4_features.csv, chart t4_segment_sizes / t4_segment_profiles / t4_silhouette

**3 luật:** (1) Wellbeing, KHÔNG credit scoring — FHS không bao giờ dùng để từ chối/giảm hạn mức/khoá. (2) So bằng tỷ lệ, không so VND tuyệt đối. (3) Mỗi nhận định có số hoặc chart.

Hướng dẫn chi tiết: `draft/YAPPERS_huong_dan_Colab.xlsx`, sheet của bạn.

In [ ]:
#@title ⚙️ SETUP — chạy 1 lần mỗi phiên Colab (không cần pip install gì cả)
# Colab có sẵn pandas / numpy / scikit-learn / matplotlib / openpyxl -> đủ cho mọi script trong draft/.
import os, sys, gzip, shutil
DRIVE_DATA = "/content/drive/MyDrive/BI10_ROUND01_DATASET"   # thư mục data trên Drive (xem sheet 'Setup Colab')
REPO = "/content/BI10GroupYAPPERS"

from google.colab import drive, userdata
drive.mount("/content/drive")

if os.path.exists(REPO):
    os.system(f"git -C {REPO} pull -q")
else:
    tok = userdata.get("GH_TOKEN")   # 🔑 Secrets bên trái: tên GH_TOKEN, bật "Notebook access"
    os.system(f"git clone -q https://{tok}@github.com/thanh-tai-435/BI10GroupYAPPERS.git {REPO}")
assert os.path.exists(REPO), "Clone thất bại: kiểm tra GH_TOKEN + đã được mời vào repo chưa"
os.chdir(REPO)

# Dựng lại đúng đường dẫn mà các script draft/*.py đang dùng
D = "BI10_ROUND01_DATASET"
TX = f"{D}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
os.makedirs(os.path.dirname(TX), exist_ok=True)
for f in ["consumer_financial_health_engagement_2025.csv", "data_dictionary.xlsx"]:
    if not os.path.exists(f"{D}/{f}") and os.path.exists(f"{DRIVE_DATA}/{f}"):
        shutil.copy(f"{DRIVE_DATA}/{f}", f"{D}/{f}")
if not os.path.exists(TX):
    gz, raw = f"{DRIVE_DATA}/consumer_transactions_2025.parquet.gz", f"{DRIVE_DATA}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
    if os.path.exists(gz):   # .parquet.gz thực chất là CSV giao dịch nén gzip (134MB -> 597MB, ~30s)
        with gzip.open(gz) as src, open(TX, "wb") as dst: shutil.copyfileobj(src, dst)
    else:                    # hoặc upload nguyên thư mục .parquet/ chứa CSV 597MB
        shutil.copy(raw, TX)

# Script gốc gọi sys.stdout.reconfigure(); stdout của Colab không có hàm này -> cho thành no-op
if not hasattr(sys.stdout, "reconfigure"):
    type(sys.stdout).reconfigure = lambda self, **k: None
os.makedirs("outputs/figures", exist_ok=True)
if not os.path.exists("draft/task4_features.csv"):   # nhãn segment (git-ignore) — task3/task5/charts cần
    os.system("python draft/task4.py > /dev/null")
print("✅ Sẵn sàng — cwd:", os.getcwd())


## Bước 0 — Chạy baseline
Nạp toàn bộ biến của script gốc (`mon`, `cons`, `txn`…) vào notebook. So kết quả với `draft/task4_output.txt`.

In [ ]:
%run -i draft/task4.py

## Việc 1 — Preprocessed Dataset (deliverable BẮT BUỘC)  `[BẮT BUỘC · hạn 27/09 12:00]`
Xuất task4_features.csv: 1 dòng/khách, 8 feature thô + 8 feature z-scale + nhãn segment + cột nhân khẩu. Kèm data_dictionary_task4.md (tên cột, ý nghĩa, công thức, đơn vị). Tải về từ Colab (thanh file bên trái) gửi P5 cho ZIP.

**Giao:** CSV + dictionary → P5

In [ ]:
# Gợi ý:
# out.to_csv('draft/task4_features.csv'); from google.colab import files; files.download('draft/task4_features.csv')


## Việc 2 — Kiểm định độ bền (SLIDE MỚI)  `[Cao · hạn 27/09 12:00]`
(a) 20 random_state → ARI so nghiệm gốc (>0.8 = ổn định). (b) Bootstrap 80% × 50 lần → % khách giữ segment. (c) GMM k=4: ARI với k-means + % khách có xác suất < 0.6 ('khách biên'). (d) PCA 2D scatter tô màu segment.

**Giao:** Chart t4_pca.png + bảng ARI, slide mới

In [ ]:
# Gợi ý:
# from sklearn.metrics import adjusted_rand_score
# [adjusted_rand_score(labels, KMeans(4, random_state=s, n_init=10).fit_predict(X)) for s in range(20)]


## Việc 3 — Dịch chuyển segment theo tháng  `[Cao · hạn 27/09 18:00]`
Gán mỗi dòng consumer-month vào centroid gần nhất (cùng scaler) → ma trận chuyển tháng t → t+1. Bao nhiêu % Healthy trượt sang Stretched trong T11–T12? → bằng chứng 'stress theo đợt' + Planning Reminder trước T12 (gửi P5).

**Giao:** Ma trận chuyển + 1 con số gửi P5

In [ ]:
# Gợi ý:
# mon['seg']=km.predict(scaler.transform(mon_feats)); pd.crosstab(prev_seg, seg, normalize='index')


## Việc 4 — Hồ sơ nhân khẩu từng segment  `[Trung bình · hạn 27/09 18:00]`
Tuổi, giới, top tỉnh theo segment → chứng minh segment không trùng nhân khẩu → công bằng. Đồng bộ số giới tính (56.6% vs 49.4%) với P5.

**Giao:** Bảng slide 15

In [ ]:
# Gợi ý:
# feat.groupby('segment')[['age']].describe(); pd.crosstab(feat.segment, feat.gender, normalize='index')


## Việc 5 — Ánh xạ sang tên gợi ý của đề  `[Trung bình · hạn 27/09 18:00]`
Bảng 1 dòng / segment: vì sao 'Healthy but Disengaged' và 'Essential-Spend-Focused' không thành cụm riêng (nhóm 25 khách của P3 nằm ở segment nào?).

**Giao:** Bảng slide 14

In [ ]:
# Gợi ý:
# —


## Việc 6 — Limitations & Future Work  `[Thấp · hạn 27/09 18:00]`
≥3 bullet trên slide 15 hoặc slide mới.

**Giao:** Bullet

In [ ]:
# Gợi ý:
# —


## Findings mới (2–4 dòng / việc, có số)
- ...

---
**Lưu bài:** `File → Save a copy in GitHub` → repo `thanh-tai-435/BI10GroupYAPPERS`, branch `main`, path `notebooks/task4_segmentation.ipynb` (mỗi người chỉ lưu notebook của mình → không conflict). Chart: `savefig('outputs/figures/tN_ten.png')` rồi tải về gửi P5.